In [ ]:
# Importing the libraries
import pandas as pd
import spacy
import re
from nltk.tokenize import sent_tokenize, word_tokenize
from spacy.lang.en.stop_words import STOP_WORDS
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk

In [ ]:
# Load the dataset

data = pd.read_csv(r"C:\Users\nuzha\Downloads\archive (5)\instagram.csv")
df = data.iloc[:1000]
df.head()

In [ ]:
df.info()


## Cleaning the texts 

In [ ]:


# Load spaCy English model
nlp = spacy.load('en_core_web_sm')

# Define text cleaning functions

def clean_text(text):
    """Clean the text by removing citations, extra whitespaces, and converting to lowercase."""
    text = re.sub(r'\[\d+\]', '', text)  # Remove citations
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra whitespaces and strip
    return text.lower()

def remove_punctuation(text):
    """Remove punctuation from the text."""
    PUNCT_TO_REMOVE = """!"'()*+,-/:;<=>?[\]_`{|}~@#$^%&\n\t"""
    return text.translate(str.maketrans('', '', PUNCT_TO_REMOVE))

def lemmatize_and_filter(text):
    """Lemmatize and filter out non-English words using spaCy."""
    doc = nlp(text)
    return ' '.join([token.lemma_ for token in doc if token.lang_ == 'en'])

def nltk_clean_and_tokenize(text):
    """Tokenize the text and clean it further."""
    sentences = sent_tokenize(text)
    return [' '.join(word_tokenize(sent)) for sent in sentences]

def process_text(review):
    """Process the text through all cleaning steps and calculate word count."""
    clean_rv = clean_text(review)
    clean_rv = remove_punctuation(clean_rv)
    clean_rv = lemmatize_and_filter(clean_rv)
    clean_rv = ' '.join([word for word in clean_rv.split() if word not in STOP_WORDS])
    cleaned_review = nltk_clean_and_tokenize(clean_rv)
    return cleaned_review

def count_word(review):
    """Count the number of words in the review."""
    return sum(len(word_tokenize(sentence)) for sentence in review)

# Apply the text processing and word count functions


df['clean_review'] = df['review_description'].apply(process_text)
df['word_count_clean'] = df['clean_review'].apply(count_word)

df['clean_review'] = df['clean_review'].apply(lambda x: ' '.join(x))


df.sample(10)


# Predicting App Review Ratings

### Feature Engineering

In [ ]:

tfidf_vector = TfidfVectorizer()
X_tfidf = tfidf_vector.fit_transform(df['clean_review'])

### Model Development

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error,mean_absolute_error
import numpy as np

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

In [ ]:


# Defining the  parameter grids for each model
param_grid_ridge = {
    'alpha': [0.1, 1.0, 10.0, 100.0]
}

param_grid_lasso = {
    'alpha': [0.1, 1.0, 10.0, 100.0]
}

param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30]
}

param_grid_gb = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 7]
}

# Define models
models = {
    'Ridge Regression': (Ridge(), param_grid_ridge),
    'Lasso Regression': (Lasso(), param_grid_lasso),
    'Random Forest': (RandomForestRegressor(random_state=42), param_grid_rf),
    'Gradient Boosting': (GradientBoostingRegressor(random_state=42), param_grid_gb)
}

# Function to perform grid search and evaluate model
def perform_grid_search(model, param_grid, X_train, y_train, X_test, y_test, model_name):
    grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=5, scoring='neg_mean_squared_error')
    grid_search.fit(X_train, y_train)
    
    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test)
    
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    
    print(f"{model_name} - Best Parameters: {grid_search.best_params_}")
    print(f"{model_name} - Best Score (neg MSE): {-grid_search.best_score_}")
    print(f"{model_name} - RMSE: {rmse}, MAE: {mae}")
    print("-------------------------------------------------")
    
    return best_model

# Splitinging data into training and test sets
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_tfidf, df['rating'], test_size=0.2, random_state=42)

# Perform grid search for each model
best_models = {}
for model_name, (model, param_grid) in models.items():
    best_model = perform_grid_search(model, param_grid, X_train, y_train, X_test, y_test, model_name)
    best_models[model_name] = best_model


**Ridge Regression outperforms the other models with the lowest RMSE and MAE, suggesting it is the most accurate for  data.**

# Anomaly Detection in Ratings

In [ ]:
param_grid = {
    'alpha': [0.1, 1.0, 10.0, 100.0]
}

ridge = Ridge()

grid_search = GridSearchCV(estimator=ridge, param_grid=param_grid, cv=5, scoring='neg_mean_squared_error')

grid_search.fit(X_train, y_train)

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best Score: {-grid_search.best_score_}")

best_ridge_model = grid_search.best_estimator_
y_pred_best_ridge = best_ridge_model.predict(X_test)
df['predicted_rating'] = best_ridge_model.predict(X_tfidf)

df['rating_difference'] = np.abs(df['rating'] - df['predicted_rating'])

threshold = 1

df['anomaly'] = df['rating_difference'] > threshold

anomalies = df[df['anomaly']]
anomalies.info()

### Highlighting Anomalies

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(df['rating'], df['predicted_rating'], c='blue', label='Normal')
plt.scatter(anomalies['rating'], anomalies['predicted_rating'], c='red', label='Anomaly')
plt.xlabel('Actual Rating')
plt.ylabel('Predicted Rating')
plt.title('Actual vs Predicted Ratings')
plt.legend()
plt.show()



In [ ]:
def export_data_to_excel(df, file_path):
     """Export the DataFrame to an Excel file."""
     with pd.ExcelWriter(file_path, engine='xlsxwriter') as writer:
         df.to_excel(writer, index=False)

# Dataset exporting 

export_data_to_excel(df, 'review_processed_data.xlsx')